# Level of Service Deepdive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AiMTT-project/UC7-SAIL/blob/main/2.2%20LOS/Assignment/los.ipynb)


Level of Service (LOS) is a commonly used framework to describe the operational quality of pedestrian facilities. It classifies conditions from **LoS A (free flow)** to **LoS F (congested or breakdown conditions)** based on key variables such as **density, flow, and walking speed**. LOS provides an intuitive way to interpret crowd conditions in terms of comfort, safety, and mobility.

The most widely used reference for pedestrian LOS is the work of John J. Fruin (1971). Fruin’s model is based on empirical observations and defines LOS primarily through **pedestrian density ranges**, with corresponding relationships to flow and speed. A key assumption in this framework is that pedestrians move in a **unidirectional flow**, such as along sidewalks or corridors, where interactions between individuals are relatively limited.

In real-world situations, however, pedestrian movement is often **bidirectional or multidirectional**, especially in open spaces, stations, or bridges. In these cases, interactions between opposing flows introduce **friction, conflicts, and lane formation**, which reduce walking speed and overall efficiency. As a result, for the same density, bidirectional flows typically experience **lower flow rates and poorer perceived conditions** than unidirectional flows.

Overall, LOS remains a valuable and widely accepted tool for assessing pedestrian conditions, but its application should always consider the **type of flow and context of the environment**.

### Import necessary libraries

In [ ]:
!pip install contextily
!pip install pandas
!pip install geopandas

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
import matplotlib.patches as mpatches
import numpy as np
import os
import imageio.v2 as imageio

### Import all necessary helper functions

---





In [ ]:
# @title
def in_range(
    value: float,
    min_val: float = None,
    max_val: float = None,
    include_min: bool = False,
    include_max: bool = True,
) -> bool:
    if pd.isna(value):
        return False

    lower_ok = (
        True
        if min_val is None
        else (value >= min_val if include_min else value > min_val)
    )
    upper_ok = (
        True
        if max_val is None
        else (value <= max_val if include_max else value < max_val)
    )

    return lower_ok and upper_ok


def classify_metric(value: float, thresholds: dict, metric: str) -> str:
    for los, bounds in thresholds.items():
        min_val = bounds[metric].get("min")
        max_val = bounds[metric].get("max")

        if in_range(value, min_val=min_val, max_val=max_val):
            return los

    return np.nan


def combine_los(los_density: str, los_flow: str) -> str:
    los_rank = {
        "A": 1,
        "B": 2,
        "C": 3,
        "D": 4,
        "E": 5,
        "F": 6,
    }

    if pd.isna(los_density) and pd.isna(los_flow):
        return np.nan
    if pd.isna(los_density):
        return los_flow
    if pd.isna(los_flow):
        return los_density

    if los_density in los_rank and los_flow in los_rank:
        return max([los_density, los_flow], key=lambda x: los_rank[x])

    if los_density in los_rank:
        return los_density
    if los_flow in los_rank:
        return los_flow

    return np.nan


def save_gdf_as_mp4(gdf: gpd.GeoDataFrame):

    # --- Settings ---
    los_order = ["LoS A", "LoS B", "LoS C", "LoS D", "LoS E", "LoS F"]

    los_colors = {
        "LoS A": "#002CCF",
        "LoS B": "#58AA2A",
        "LoS C": "#F8EC39",
        "LoS D": "#F1C73A",
        "LoS E": "#E56F2F",
        "LoS F": "#8A3817",
    }

    # --- Prepare data ---
    gdf_day = gdf.copy()
    gdf_day["time"] = pd.to_datetime(gdf_day["time"])
    gdf_day["time_10min"] = gdf_day["time"].dt.floor("10min")
    gdf_day["LoS"] = pd.Categorical(gdf_day["LoS"], categories=los_order, ordered=True)
    gdf_day["color"] = gdf_day["LoS"].map(los_colors)

    # Optional: keep only one day
    gdf_day = gdf_day[
        gdf_day["time"].dt.date == pd.Timestamp("2025-08-20").date()
    ].copy()

    # Reproject once
    gdf_day = gdf_day.to_crs(epsg=3857)

    # Fixed extent
    xmin, ymin, xmax, ymax = gdf_day.total_bounds

    # Ordered timestamps
    time_steps = sorted(gdf_day["time_10min"].dropna().unique())

    # Legend
    legend_patches = [
        mpatches.Patch(color=los_colors[los], label=los) for los in los_order
    ]

    # Output folder
    frame_dir = "los_frames"
    os.makedirs(frame_dir, exist_ok=True)

    frame_paths = []

    # --- Save each frame as PNG ---
    for i, current_time in enumerate(time_steps):
        gdf_step = gdf_day[gdf_day["time_10min"] == current_time].copy()

        fig, ax = plt.subplots(figsize=(10, 10))

        if not gdf_step.empty:
            gdf_step.plot(
                ax=ax,
                color=gdf_step["color"],
                alpha=0.7,
                edgecolor="black",
                linewidth=0.8,
            )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

        # Add basemap after setting extent
        ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

        ax.legend(handles=legend_patches, title="Level of Service", loc="upper right")
        ax.set_title(
            f"Level of Service at {pd.Timestamp(current_time).strftime('%Y-%m-%d %H:%M')}"
        )
        ax.set_axis_off()

        frame_path = os.path.join(frame_dir, f"frame_{i:03d}.png")
        plt.savefig(frame_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        frame_paths.append(frame_path)

    # --- Combine PNGs into MP4 ---
    with imageio.get_writer("los_animation.mp4", fps=2) as writer:
        for frame_path in frame_paths:
            writer.append_data(imageio.imread(frame_path))
    os.system(f"rm {frame_dir}/*.png")  # Clean up frames
    print("Saved: los_animation.mp4")


### Retrieve necessary data

Add the dataset "LOS_data_oefening1_english.geojson" from the practice data folder to you sample_data folder under files in Google Colab.

In [ ]:
pedestrian_count_path = "/content/sample_data/LOS_deepdive_data.geojson"


In [ ]:
gdf = gpd.read_file(pedestrian_count_path)


### Understanding the data

First off, we are going to perform some basic exploration of the data we'll be working with. To start, lets explore which attributes the dataset has.

In [ ]:
# Print all columns in gdf and their data types in a clear table
df_gdf_info = pd.DataFrame({
    "Attribute": gdf.columns,
    "Data Type": [gdf[col].dtype for col in gdf.columns]
})
print(df_gdf_info.to_string(index=False))

So we are working with 7 attributes. The dataset consists of two counts, for a sensor and location set over a timeperiod. The count_line is the amount of people measured walking over a measuring line tracked by the sensor. The count_area is the amount of people measured during the same period in the area the sensor is focussed on.

---



In [ ]:
gdf.head(2)

Furthermore, we are working with a dataset which tracks counts for multiple sensors and locations over a period of time. Lets see what we can find out about the time granularity and extent of the geodataframe.

In [ ]:
# Print start and end time, and time granularity stats
start_time = gdf['time'].min()
end_time = gdf['time'].max()

print(f"Start time: {start_time}")
print(f"End time: {end_time}")
# Show the most common and average time between datapoints in seconds
# Calculate and print the average time between datapoints, grouped by location
gdf['time'] = pd.to_datetime(gdf['time'])
avg_time_by_location = (
    gdf.sort_values(['location', 'time'])
    .groupby('location')['time']
    .apply(lambda x: x.diff().dropna().mean())
)
print("Average time between datapoints by location:")
print(avg_time_by_location)

This shows that we are working with data from 20 August between 08:00 and 16:30. It also indicates that the dataset includes multiple locations with a time granularity of approximately 1.5 to 2 minutes. Let's explore which locations we'll be working with.

In [ ]:

# Plot all areas using their geometry on an OpenStreetMap basemap
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, column='location', legend=True, alpha=0.5, edgecolor='k')
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, crs=gdf.crs)
ax.set_title("All Areas by Location on OpenStreetMap")
plt.axis('off')
plt.show()

So we are exploring multiple areas, mostly bridges around Central Station in Amsterdam on the 20th of August, the first day of SAIL 2025. Finally, let's examine the actual counts the sensored measured on this day.

In [ ]:
# Plot count_line for each location over time
fig, ax = plt.subplots(figsize=(12, 6))
for location in gdf['location'].unique():
    subset = gdf[gdf['location'] == location]
    ax.plot(pd.to_datetime(subset['time']), subset['count_line'], label=location)
ax.set_xlabel('Time')
ax.set_ylabel('Count Line')
ax.set_title('Pedestrian Count Line per Location Over Time')
ax.legend()
plt.show()

This dataset reveals a clear trend that can be analysed across all locations. However, during a live event, such data is not immediately available or sufficiently actionable in real time. To address this limitation, a Level of Service (LOS) approach can be applied.

To determine the Level of Service of a given area, both density and flow must be considered. Flow, as defined by John J. Fruin, represents the number of pedestrians passing a specific point per unit of time, typically expressed per metre of width per minute. Density we can define as the amount of people per square meter.

Both these attributes can be calculated with the given data.

*Assignment 1*: In the code block below finish the calculation for density and flow.

In [ ]:
gdf["density"] = ##TODO calculate density
gdf["flow"] = ##TODO calculate flow

### Different approaches

There are multiple ways to define Level of Service. In this study, we consider LOS classifications for both unidirectional and bidirectional pedestrian flows. These approaches are applied both generally across all areas and specifically to different types of locations.

The first approach we are going to look at is a general LOS classification for all areas. Here we take a look at LOS thresholds as determined by Fruin (1971) for unidirectional flow.

![alt text](https://www.gkstill.com/_Media/fruin_med_hr.jpeg)

On the first day of SAIL 2025, "SAIL IN" occured. During SAIL IN, between 13.00 and 14.00 all the tallships enter the the IJgebied around central station. This of course has an effect on pedestrian crowd behaviour.

*Assignment 2:* Filter the dataset on a minute 13.30 on the 20th of August. Calculate the level of service according to the LOS thresholds above. Make sure to name the geodataframe gdf_minute, the attribute column "LoS" and each value "LoS A" up to "LoS F". This is important for visualisation later.

In [ ]:
gdf_minute = ""

#### Visualisation

In [ ]:

# Make sure LoS is ordered for consistent plotting
los_order = ["LoS A", "LoS B", "LoS C", "LoS D", "LoS E", "LoS F"]
gdf_minute["LoS"] = pd.Categorical(gdf_minute["LoS"], categories=los_order, ordered=True)

# Fixed Fruin-style color scheme
los_colors = {
    "LoS A": "#002CCF",
    "LoS B": "#58AA2A",
    "LoS C": "#F8EC39",
    "LoS D": "#F1C73A",
    "LoS E": "#E56F2F",
    "LoS F": "#8A3817",
}
# Assign a color to each row
gdf_minute["color"] = gdf_minute["LoS"].map(los_colors)

# Reproject to Web Mercator for contextily
gdf_plot = gdf_minute.to_crs(epsg=3857)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

gdf_plot.plot(
    ax=ax,
    color=gdf_plot["color"],
    alpha=0.7,
    edgecolor="black",
    linewidth=0.8
)

ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

# Custom legend in correct order
legend_patches = [
    mpatches.Patch(color=los_colors[los], label=los)
    for los in los_order
    if los in gdf_minute["LoS"].dropna().unique()
]
ax.legend(handles=legend_patches, title="Level of Service", loc="upper right")

ax.set_title("Level of Service on 2025-08-20 between 15:30 and 15:31")
ax.set_axis_off()
plt.tight_layout()
plt.show()

The areas visualized above did not actually contain unidirectional pedestrian flows, but bidirectional flows. So this means pedestrians can enter the LOS area from two sides, which has an effect on the relation between flow and density.

*Assignment 3.1:* Fill the table and mapping below with general density and flow thresholds which you deem correct for the bidirectional flow areas the data contains. You can use the the code already written code block below to visualize your settings.

In [ ]:
 ## TODO provide a table with variable names, but empty values. Also provide mapping for the key and values
 # Create a table with variable names and empty values for density and flow thresholds
los_thresholds = {
    "A": {"density": {"min": None, "max": None}, "flow": {"min": None, "max": None}},
    "B": {"density": {"min": None, "max": None}, "flow": {"min": None, "max": None}},
    "C": {"density": {"min": None, "max": None}, "flow": {"min": None, "max": None}},
    "D": {"density": {"min": None, "max": None}, "flow": {"min": None, "max": None}},
    "E": {"density": {"min": None, "max": None}, "flow": {"min": None, "max": None}},
    "F": {"density": {"min": None, "max": None}, "flow": {"min": None, "max": None}},
}

In [ ]:
gdf_visualisation = gdf.copy()
gdf_visualisation["los_density"] = gdf_visualisation["density"].apply(
    lambda x: classify_metric(x, los_thresholds, "density")
)
gdf_visualisation["los_flow"] = gdf_visualisation["flow"].apply(
    lambda x: classify_metric(x, los_thresholds, "flow")
)
# Compute combined LoS once
gdf_visualisation["los_combined"] = gdf_visualisation.apply(
    lambda row: combine_los(row["los_density"], row["los_flow"]),
    axis=1
)
# Create final visualization column
gdf_visualisation["LoS"] = gdf_visualisation["los_combined"].apply(
    lambda x: f"LoS {x}" if pd.notna(x) else np.nan
)

gdf_minute = gdf_visualisation[(gdf_visualisation['time'] >= '2025-08-20 15:30:00+02:00') & (gdf_visualisation['time'] < '2025-08-20 15:31:00+02:00')]# Make sure LoS is ordered for consistent plotting


#### Visualisation

In [ ]:
los_order = ["LoS A", "LoS B", "LoS C", "LoS D", "LoS E", "LoS F"]

gdf_minute["LoS"] = pd.Categorical(gdf_minute["LoS"], categories=los_order, ordered=True)

# Fixed Fruin-style color scheme
los_colors = {
    "LoS A": "#002CCF",
    "LoS B": "#58AA2A",
    "LoS C": "#F8EC39",
    "LoS D": "#F1C73A",
    "LoS E": "#E56F2F",
    "LoS F": "#8A3817",
}
# Assign a color to each row
gdf_minute["color"] = gdf_minute["LoS"].map(los_colors)

# Reproject to Web Mercator for contextily
gdf_plot = gdf_minute.to_crs(epsg=3857)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

gdf_plot.plot(
    ax=ax,
    color=gdf_plot["color"],
    alpha=0.7,
    edgecolor="black",
    linewidth=0.8
)

ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

# Custom legend in correct order
legend_patches = [
    mpatches.Patch(color=los_colors[los], label=los)
    for los in los_order
    if los in gdf_minute["LoS"].dropna().unique()
]
ax.legend(handles=legend_patches, title="Level of Service", loc="upper right")

ax.set_title("Level of Service on 2025-08-20 between 15:30 and 15:31")
ax.set_axis_off()
plt.tight_layout()
plt.show()

*Assignment 3.2:* Below is a indication of a general LOS classification for bidirectional areas. Apply the values in the previous assignment and reflect on how they differ from you own approach:

| LOS | Density (p/m²) | Flow (p/m/min) |
|-----|---------------|----------------|
| A   | < 0.3         | < 15           |
| B   | 0.3 – 0.7     | 15 – 25        |
| C   | 0.7 – 1.3     | 25 – 35        |
| D   | 1.3 – 2.0     | 20 – 30        |
| E   | 2.0 – 3.0     | 15 – 25        |
| F   | > 3.0         | < 15           |

*Assignment 4*: We have been applying a general LOS approach for all areas. Apply a more specific LOS for similar areas. For example one LOS approach for all bridges and one for the area around Central Station.

Again make sure to name the geodataframe "gdf_visualisation", the column for the classification "LoS" and the values "LoS A" up to "LoS F" for visualisation purposes.

#### Visualisation

If you have named your geodataframe "gdf_visualisation" and have assigned the correct attribute and value names, you can generate a MP4 of the progression of the LoS classification of all areas.

In [ ]:
save_gdf_as_mp4(gdf_visualisation)

## Conclusion

What is important to take away from this deepdive, is that not one event or area will necessarly use the same LOS thresholds. It is up to crowd managers to take into account a range of factors to determine what good LOS definitions are to use for an area and event. Factors which we discussed in this deepdive, such as uni/bidirectional flow, the type of event and it's visitorgroup.